# IndicVAD-Bench on Colab

**Runtime → Change runtime type → T4 GPU** before running anything.

Storage split (important):
| What | Where | Survives disconnect? |
|---|---|---|
| Downloaded archives | Drive `00_archives/` | ✅ yes |
| Extracted audio, pool, posteriors | local `/content` scratch | ❌ no |
| Results + paper pack | Drive `05_results/`, `06_paper_pack/` | ✅ yes |

Drive is a FUSE mount — fast for a few big files, terrible for 20k small ones.
Archives go to Drive so a disconnect never costs a re-download; re-extracting
to scratch takes ~2 min.

**A GPU does not speed up the default systems.** energy/ltsd/webrtc/silero are
CPU-bound by design. The GPU exists to make `marblenet` and `pyannote`
affordable — two strong baselines that materially strengthen §4. Enable them
in cell 5.

## 1. Mount Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!df -h /content | tail -1
!free -g | head -2

## 2. Get the code

Upload `indicvad.tar.gz` to your Drive root, or clone from git if you've pushed it.

In [ ]:
import os, pathlib, shutil

DRIVE = '/content/gdrive/MyDrive'
os.makedirs(f'{DRIVE}/indicvad', exist_ok=True)

# Option A: extract the tarball you uploaded to Drive
if pathlib.Path(f'{DRIVE}/indicvad.tar.gz').exists():
    !tar -xzf "{DRIVE}/indicvad.tar.gz" -C /content/
    %cd /content/indicvad
# Option B: clone your repo
# !git clone https://github.com/YOU/indicvad /content/indicvad
# %cd /content/indicvad

!ls

## 3. Install dependencies (~4 min)

In [ ]:
# torch + CUDA already present on Colab GPU runtimes
!pip install -q -r requirements.txt
!pip install -q onnxruntime silero-vad webrtcvad statsmodels huggingface_hub

# Optional heavy baselines -- these are the ones that use the GPU.
# pyannote is a ~200 MB install; nemo is ~2 GB and slow. Enable what you need.
!pip install -q "pyannote.audio>=3.1"
# !pip install -q "nemo_toolkit[asr]"

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 4. Verify the install — run this before touching real data

37 offline known-answer tests: I/O, SNR mixing, RIR delay compensation,
decoding, every metric, the variance decomposition against a planted effect,
and the phonology extractor. Takes ~20 seconds.

In [ ]:
!python selftest.py

## 5. Configure

Three things to set:

1. **`ntfy_topic`** — pick any hard-to-guess string, then open
   `https://ntfy.sh/<that string>` on your phone. You'll get a push
   notification as each step finishes, so you can walk away.
2. **`hf_token`** — only needed for pyannote. Get one at
   huggingface.co/settings/tokens, **and accept the model terms** at
   huggingface.co/pyannote/segmentation-3.0 (the download fails silently
   otherwise).
3. **Which systems to enable.**

In [ ]:
import yaml, secrets

cfg = yaml.safe_load(open('config.yaml'))

NTFY_TOPIC = 'indicvad-' + secrets.token_hex(4)   # or set your own string
HF_TOKEN   = ''                                    # paste yours for pyannote

cfg['storage'] = {'mode': 'colab',
                  'drive_root': '/content/gdrive/MyDrive/indicvad',
                  'scratch_root': '/content/indicvad_scratch'}
cfg['runtime']['device'] = 'auto'
cfg['credentials']['ntfy_topic'] = NTFY_TOPIC
cfg['credentials']['hf_token'] = HF_TOKEN

# GPU-worthwhile systems. pyannote needs a token; marblenet needs nemo.
cfg['systems']['pyannote']['enabled']  = bool(HF_TOKEN)
cfg['systems']['marblenet']['enabled'] = False   # flip to True if nemo installed

yaml.safe_dump(cfg, open('config.yaml', 'w'), sort_keys=False)

print('Open this on your phone for push notifications:')
print(f'   https://ntfy.sh/{NTFY_TOPIC}')
print()
print('enabled systems:', [k for k, v in cfg['systems'].items() if v['enabled']])

## 6. Start the human-gold annotation pack — do this now, not later

This is a **standalone step** with no dependency on the rest of the pipeline.
The real-audio half needs only the config you just wrote and network access —
not Step 01, not a GPU, nothing else on this page. Kick it off immediately so
an annotator has real clips in hand while everything below is still running.
Re-run this same cell after Step 03 (§8) and the synthetic half fills in too;
already-fetched real clips are never re-downloaded.

In [ ]:
!python main.py --steps 03b
# Check human_gold/INSTRUCTIONS.txt for which languages got real clips --
# a bad config name or missing network access is reported per-language,
# not as a pipeline failure. Re-run after fixing access or after Step 03.

## 7. Preflight

Check the plan, the device, disk space and the runtime budget before committing.

In [ ]:
!python main.py --dry-run

## 8. Smoke test first (~15 min)

Do **not** launch the full grid blind. Run a tiny config end to end, confirm
the CSVs look sane, then scale up. This has caught more wasted overnight runs
than any other habit.

In [ ]:
import yaml, copy
full = yaml.safe_load(open('config.yaml'))
smoke = copy.deepcopy(full)
smoke['languages'] = [l for l in full['languages']
                      if l['code'] in ('hi_in', 'ta_in', 'en_us')]
smoke['benchmark']['sessions_per_lang'] = 6
smoke['benchmark']['session_duration_s'] = 20
smoke['data']['max_utts_per_lang'] = 60
smoke['conditions'] = {'snr_db': [10, 0], 'noise_types': ['pointsource'],
                       'reverb': ['none'], 'babble_n_talkers': 8}
smoke['systems'] = {k: dict(v, enabled=(k in ('energy', 'webrtc')))
                    for k, v in full['systems'].items()}
smoke['stats'] = {'n_bootstrap': 200, 'n_permutation': 200, 'alpha': 0.05}
yaml.safe_dump(smoke, open('config_smoke.yaml', 'w'), sort_keys=False)
print('wrote config_smoke.yaml')

In [ ]:
!python main.py --config config_smoke.yaml

## 9. Full run

Steps 01–03 first (download + build). Archives land on Drive, so this is the
one-time cost — a disconnect after this point never re-downloads.

In [ ]:
!python main.py --steps 01 02 03
!python main.py --steps 03b   # re-run: synthetic half now fills in

### Step 04 — the long one

This dominates wall-clock. Progress is checkpointed per (system, condition),
so a disconnect loses at most one condition. If you get disconnected, just
re-run this same cell: completed conditions are skipped.

Throughput (`Nx realtime`) is printed per condition, with a running ETA.

In [ ]:
!python main.py --steps 04

### Steps 05–08 — analysis (fast, all CPU)

These run on cached posteriors, so they're minutes regardless of GPU.

Step 08 no longer touches the annotation pack -- see §6. Once your annotator returns labels, score them separately:

In [ ]:
!python main.py --steps 05 06 07 08

In [ ]:
!python main.py --steps 03b --score-human-gold

## 10. Full RQ3 — transfer geometry (optional, GPU)

Frozen WavLM features are extracted **once** (step 09), then every transfer
experiment (step 10) trains a small head on cached tensors and costs minutes.

| Component | T4 time |
|---|---|
| Feature extraction, 4-condition subset | ~25 min |
| Layer probe | ~10 min |
| Adaptation curves + LOLO + LOFO | ~30 min |

LoRA fine-tuning of the encoder is **not** implemented — ~120 encoder runs
(20+ GPU-h) exceeds a free allocation and wouldn't change the qualitative
answer.

Remember the page budget: this is worth **one** figure in the paper. See
README §7.

In [ ]:
!pip install -q transformers
# Optional: URIEL typological vectors. Falls back to config attributes if absent.
# !pip install -q lang2vec

import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['ssl']['enabled'] = True
cfg['ssl']['layer'] = 'auto'          # runs the layer probe, then caches that layer
yaml.safe_dump(cfg, open('config.yaml', 'w'), sort_keys=False)
print('ssl:', cfg['ssl'])

In [ ]:
!python main.py --steps 09      # extract + cache features (the GPU part)

In [ ]:
!python main.py --steps 10 11   # transfer curves, then rebuild the paper pack

**Reading the output**

- `ssl_layer_probe.csv` — if the best layer isn't the last, say so in §4.
  Most papers take the final hidden state without checking.
- `transfer_curves.csv` — two headline numbers: the unseen-language penalty
  (`lolo` − `matched`) and the extra cost of crossing a family boundary
  (`lofo` − `lolo`).
- Gap closure **above 100%** is real, not a bug: a target-adapted specialist
  can beat the matched multilingual generalist. Report it.
- The typology correlation may come back **undefined** — with a balanced
  two-family design every language sits at the same mean distance from the
  rest, so the predictor has no variance. Step 10 says so explicitly rather
  than printing a bare `nan`.

## 11. Collect results

Everything precious is already on Drive. This just lists it and zips a copy
you can download to your laptop.

In [ ]:
DRIVE_OUT = '/content/gdrive/MyDrive/indicvad'
!ls -la "{DRIVE_OUT}/06_paper_pack/"
!cat "{DRIVE_OUT}/06_paper_pack/SUMMARY.md"

import shutil
shutil.make_archive('/content/paper_pack', 'zip', f'{DRIVE_OUT}/06_paper_pack')
from google.colab import files
files.download('/content/paper_pack.zip')

## If you get disconnected

| Lost | Recover by |
|---|---|
| Downloaded archives | Nothing — they're on Drive |
| Extracted audio / pool / benchmark | `!python main.py --steps 01 02 03` (~15 min, no re-download) |
| Posteriors (step 04) | `!python main.py --steps 04` — completed conditions are skipped |
| Results / paper pack | Nothing — they're on Drive |

**Keeping the session alive:** Colab disconnects after ~30 min of *browser*
idleness, not compute idleness. A running cell keeps the kernel busy, but the
tab must stay open. Leave it open on your phone or laptop, and rely on the
ntfy notifications rather than watching it.

**Free-tier limits:** roughly 8 GPU-hours/day, and Colab may reclaim your GPU
under load. The full pipeline fits comfortably; if you get bumped to CPU
mid-run, step 04 still completes — just slower — because the wrappers fall
back automatically.